In [1]:
import numpy as np
import pandas as pd

In [7]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 50000

states = {
    "CA": {
        "Los Angeles": [("Los Angeles", "90001"), ("Pasadena", "91101"), ("Long Beach", "90802")],
        "Orange": [("Irvine", "92602"), ("Anaheim", "92801"), ("Santa Ana", "92701")],
        "San Diego": [("San Diego", "92101"), ("Chula Vista", "91910"), ("Escondido", "92025")]
    },
    "TX": {
        "Dallas": [("Dallas", "75201"), ("Irving", "75038"), ("Garland", "75040")],
        "Harris": [("Houston", "77002"), ("Pasadena", "77502"), ("Baytown", "77520")],
        "Tarrant": [("Fort Worth", "76102"), ("Arlington", "76010"), ("Mansfield", "76063")]
    },
    "FL": {
        "Miami-Dade": [("Miami", "33101"), ("Hialeah", "33010"), ("Homestead", "33030")],
        "Orange": [("Orlando", "32801"), ("Winter Park", "32789"), ("Apopka", "32703")],
        "Broward": [("Fort Lauderdale", "33301"), ("Hollywood", "33019"), ("Pembroke Pines", "33028")]
    }
}

def sample_location(states_dict):
    state = np.random.choice(list(states_dict.keys()))
    county = np.random.choice(list(states_dict[state].keys()))
    city, zip_code = states_dict[state][county][
        np.random.randint(0, len(states_dict[state][county]))
    ]
    return state, county, city, zip_code

locations = [sample_location(states) for _ in range(n)]
state_col = [x[0] for x in locations]
county_col = [x[1] for x in locations]
city_col = [x[2] for x in locations]
zip_col = [x[3] for x in locations]

fico = np.clip(np.random.normal(725, 45, n), 580, 850).round(0).astype(int)
ltv = np.clip(np.random.normal(72, 12, n), 30, 97).round(3)
cltv = np.clip(ltv + np.random.normal(2, 3, n), ltv, 100).round(3)
loan_amount = np.random.randint(50000, 1250000, n).astype(float)
dti = np.clip(np.random.normal(36, 8, n), 8, 57).round(2)
term = np.random.choice([10, 15, 20, 25, 30], n)

purpose = np.random.choice(
    ["purchase", "rate_term_refi", "cash_out_refi"],
    size=n,
    p=[0.45, 0.35, 0.20]
)

occupancy = np.random.choice(
    ["primary", "second_home", "investment"],
    size=n,
    p=[0.75, 0.10, 0.15]
)

property_type = np.random.choice(
    ["single_family", "condo", "townhouse", "PUD", "2-4 unit"],
    size=n,
    p=[0.40, 0.15, 0.25, 0.15, 0.05]
)

channel = np.random.choice(
    ["retail", "correspondent", "broker"],
    size=n,
    p=[0.80, 0.01, 0.19]
)

fico_adj = np.where(fico >= 780, 0.15,
            np.where(fico >= 740, 0.05,
            np.where(fico >= 700, -0.05,
            np.where(fico >= 680, -0.10,
            np.where(fico >= 660, -0.15, -0.45)))))

ltv_adj = np.where(ltv <= 60, 0.10,
           np.where(ltv <= 75, 0.00,
           np.where(ltv <= 80, -0.10,
           np.where(ltv <= 90, -0.30, -0.55))))

occupancy_adj = np.where(occupancy == "primary", 0.00,
                 np.where(occupancy == "second_home", -0.20, -0.45))

purpose_adj = np.where(purpose == "purchase", 0.05,
               np.where(purpose == "rate_term_refi", 0.00, -0.15))

property_adj = np.where(property_type == "single_family", 0.00,
                np.where(property_type == "condo", -0.10,
                np.where(property_type == "townhouse", -0.05,
                np.where(property_type == "PUD", -0.03, -0.20))))

buy_side_adjustments = (
    fico_adj +
    ltv_adj +
    occupancy_adj +
    purpose_adj +
    property_adj
).round(3)

noise = np.random.normal(0, 0.08, n).round(3)
buy_side_total_price = (100 + buy_side_adjustments + noise).round(3)

loan_number = [f"LN{100000 + i}" for i in range(n)]

df = pd.DataFrame({
    "loan_number": loan_number,
    "fico": fico,
    "ltv": ltv,
    "cltv": cltv,
    "dti": dti,
    "loan_amount": loan_amount,
    "term": term,
    "state": state_col,
    "county": county_col,
    "city": city_col,
    "zip": zip_col,
    "purpose": purpose,
    "occupancy": occupancy,
    "property_type": property_type,
    "channel": channel,
    "buy_side_adjustments": buy_side_adjustments,
    "buy_side_total_price": buy_side_total_price
})

print(df.head())
print(df.describe(include="all"))

df.to_parquet("mortgage_data.parquet", index=False)

  loan_number  fico     ltv    cltv    dti  loan_amount  term state  \
0    LN100000   739  46.716  49.833  46.73     220314.0    10    FL   
1    LN100001   788  75.262  80.296  41.49     337228.0    30    FL   
2    LN100002   670  74.560  79.840  32.55     358983.0    25    FL   
3    LN100003   675  90.871  96.585  44.04     140021.0    20    FL   
4    LN100004   754  80.432  80.432  26.88     597811.0    25    CA   

       county            city    zip         purpose   occupancy  \
0  Miami-Dade       Homestead  33030        purchase  investment   
1  Miami-Dade           Miami  33101   cash_out_refi     primary   
2      Orange          Apopka  32703        purchase     primary   
3     Broward  Pembroke Pines  33028  rate_term_refi     primary   
4   San Diego     Chula Vista  91910   cash_out_refi     primary   

   property_type channel  buy_side_adjustments  buy_side_total_price  
0  single_family  retail                 -0.35                99.530  
1  single_family  reta